In [98]:
##### Generate correlelograms to empirically find threshold of within-country spatial autocorrelation

import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
from pathlib import Path


In [99]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import model data
capital_model = pd.read_csv(f"{cd}/Data/Clean/Training_data/capital_relative_final.csv")
labor_model = pd.read_csv(f"{cd}/Data/Clean/Training_data/labor_relative_final.csv")

# Import lat/lon data
coords = pd.read_csv(f"{cd}/Data/Clean/Predictors/Vectors/lat_lon.csv")

# Set file path to figure repo
fd = "/Users/carinamanitius/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/OneDrive/Dissertation/Chapter 1/Figures/corrolelograms"

In [100]:
##### Define vars
PREDICTOR_VARS = [
    'rtv_log_SOC', 'rtv_log_average_travel_time_city',
    'rtv_log_average_travel_time_port',
    'rtv_log_growing_season_length_days', 'rtv_log_GDP_pc', 'rtv_log_slope',
    'rtv_log_crop_intensity',
    'rtv_log_USD_production_per_million_HA',
    'rtv_log_tonnes_production_per_million_HA',
    'rtv_log_pop_density_people_per_100_km2',
    'rtv_log_buffalo_density_per_100_km2',
    'rtv_log_chicken_density_per_100_km2',
    'rtv_log_cattle_density_per_100_km2',
    'rtv_log_goats_density_per_100_km2', 'rtv_log_pigs_density_per_100_km2',
    'rtv_log_sheep_density_per_100_km2',
    'rtv_log_livestock_density_LU_per_100_km2',
    'rtv_log_female_share_base_100',
    'rtv_log_cereals_share_base_100_production_USD',
    'rtv_log_fibres_share_base_100_production_USD',
    'rtv_log_fruits_share_base_100_production_USD',
    'rtv_log_oilcrops_share_base_100_production_USD',
    'rtv_log_pulses_share_base_100_production_USD',
    'rtv_log_roots_tubers_share_base_100_production_USD',
    'rtv_log_rest_of_crops_share_base_100_production_USD',
    'rtv_log_sugar_crops_share_base_100_production_USD',
    'rtv_log_vegetables_share_base_100_production_USD',
    'rtv_log_rubber_share_base_100_production_USD',
    'rtv_log_ruminants_share_base_100_production_USD',
    'rtv_log_monogastrics_share_base_100_production_USD',
    'rtv_log_poultry_share_base_100_production_USD',
    'rtv_log_child_dependency_ratio_per_hundred',
    'rtv_log_pct_base_100_GDP_ag', 'rtv_log_share_base_100_large_field',
    'rtv_log_share_base_100_medium_field',
    'rtv_log_share_base_100_small_field',
    'rtv_log_share_base_100_vsmall_field',
    'rtv_log_share_base_100_with_nightlights',
    'rtv_log_pct_base_100_cropland_irrigated',
    'rtv_probability_survival_land_use_objective',
    'rtv_probability_economic_land_use_objective'
]

CAPITAL_VARS = ['rtv_log_capital_intensity_USD_per_million_USD', 'rtv_log_capital_intensity_USD_per_million_tonne']
LABOR_VARS = ['rtv_log_labor_intensity_jobs_per_million_USD', 'rtv_log_labor_intensity_jobs_per_million_tonne']

In [101]:
##### CONFIG

version = 'labor_target' # capital_target, capital_predictor, labor_target, labor_predictor
df = labor_model.copy() # capital_model, labor_model
FEATURES_TO_TEST = LABOR_VARS # CAPITAL_VARS, LABOR_VARS, PREDICTOR_VARS

### options
MIN_REGIONS_PER_COUNTRY = 5  # countries with fewer regions than this are skipped (Moran's I is too noisy)
MAX_DIST_KM = 800      # max distance to compute correlogram out to
N_BINS = 50             # number of distance bins
CUTOFF_THRESHOLD = 0.1  # correlation level considered "near independent"
Y_AXIS_RANGE = (-0.4, 0.8) 

### add lat/lon
df = df.merge(coords, on='PROJ_ID', how='left')

In [102]:
##### Define core functions

# function to calculate the haversine distance (km) between arrays of lat/lon points
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

# function to get the full distance matrix (km) for all sets of points
def pairwise_distance_matrix(lats, lons):
    lat_i = lats.values[:, None]
    lon_i = lons.values[:, None]
    lat_j = lats.values[None, :]
    lon_j = lons.values[None, :]
    return haversine_km(lat_i, lon_i, lat_j, lon_j)

# function to calculate the Moran's I within each distance bin
def morans_i_by_distance_bin(values, dist_matrix, bin_edges):
    x = values.values.astype(float)
    x = x - np.nanmean(x)
    n = len(x)
    denom = np.nansum(x ** 2)

    results = []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        w = (dist_matrix >= lo) & (dist_matrix < hi)
        np.fill_diagonal(w, False)
        w_sum = w.sum()
        if w_sum == 0 or denom == 0:
            results.append(np.nan)
            continue
        num = np.nansum(w * np.outer(x, x))
        I = (n / w_sum) * (num / denom)
        results.append(I)
    return np.array(results)

# function to comput the correlogram for each feature for each country individually 
def compute_country_correlogram(df, country, features, max_dist_km, n_bins):
    sub = df[df['country_ID'] == country].dropna(subset=['lat', 'lon'])
    n_regions = len(sub)
    if n_regions < 5:
        print(f"  [{country}] skipped — only {n_regions} regions with coords")
        return None

    dist_matrix = pairwise_distance_matrix(sub['lat'], sub['lon'])
    bin_edges = np.linspace(0, max_dist_km, n_bins + 1)
    bin_mids = (bin_edges[:-1] + bin_edges[1:]) / 2

    out_rows = []
    for feature in features:
        if feature not in sub.columns:
            print(f"  [{country}] feature '{feature}' not found — skipping")
            continue
        vals = sub[feature]
        if vals.isna().all():
            continue
        corrs = morans_i_by_distance_bin(vals, dist_matrix, bin_edges)
        for mid, c in zip(bin_mids, corrs):
            out_rows.append({
                "country": country,
                "feature": feature,
                "distance_km": mid,
                "morans_i": c,
                "n_regions": n_regions,
            })
    return pd.DataFrame(out_rows)

# function to plot correlograms for ALL features together as subplots on one figure
def plot_all_correlograms(combined_df, features):
    n_features = len(features)
    n_cols = 3
    n_rows = int(np.ceil(n_features / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4.5 * n_rows), squeeze=False)

    for i, feature in enumerate(features):
        ax = axes[i // n_cols][i % n_cols]
        feature_df = combined_df[combined_df["feature"] == feature]

        if feature_df.empty:
            ax.set_visible(False)
            continue

        pivot = feature_df.pivot_table(index="distance_km", columns="country", values="morans_i")

        for country in pivot.columns:
            ax.plot(pivot.index, pivot[country], color="gray", alpha=0.4, linewidth=1)

        mean_line = pivot.mean(axis=1, skipna=True)
        ax.plot(mean_line.index, mean_line.values, color="black", linewidth=2.5, label="mean")

        ax.set_ylim(Y_AXIS_RANGE)
        ax.set_xlabel("Distance (km)")
        ax.set_ylabel("Spatial Correlation")
        ax.set_title(feature, fontsize=10)
        ax.legend(fontsize=7, loc="upper right")

    # hide any unused subplot axes if features don't fill the grid evenly
    for j in range(n_features, n_rows * n_cols):
        axes[j // n_cols][j % n_cols].set_visible(False)

    fig.suptitle("Spatial correlograms across features\n(grey = individual countries, black = mean)", fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(f"{fd}/correlogram_all_features_{version}.png", dpi=150)
    plt.close(fig)

# functino to suggest what distance the cuttoff should be (returns nan if never drops below threshold)
def suggest_cutoff_from_series(distance_km, values, threshold):
    order = np.argsort(distance_km)
    d = np.array(distance_km)[order]
    v = np.array(values)[order]
    below = v < threshold
    idx = np.argmax(below) if below.any() else None
    if idx is not None and below[idx]:
        return d[idx]
    return np.nan

In [103]:
missing_cols = [c for c in ['country_ID', 'lat', 'lon'] if c not in df.columns]
if missing_cols:
    raise ValueError(
        f"Missing expected columns: {missing_cols}. "
    )

all_countries = sorted(df['country_ID'].dropna().unique().tolist())
print(f"Found {len(all_countries)} countries in data.")

all_results = []

for country in all_countries:
    n_regions = df[df['country_ID'] == country].dropna(subset=['lat', 'lon']).shape[0]
    if n_regions < MIN_REGIONS_PER_COUNTRY:
        print(f"  [{country}] skipped — only {n_regions} regions with coords")
        continue

    print(f"  [{country}] computing correlogram ({n_regions} regions)...")
    country_df = compute_country_correlogram(
        df, country, FEATURES_TO_TEST, MAX_DIST_KM, N_BINS
    )
    if country_df is not None and not country_df.empty:
        all_results.append(country_df)

if not all_results:
    print("\nNo results generated — check coordinate columns and country codes.")
else:
    combined = pd.concat(all_results, ignore_index=True)
    combined.to_csv(f'{fd}/correlogram_values_{version}.csv', index=False)

    # one combined figure with all features as subplots
    plot_all_correlograms(combined, FEATURES_TO_TEST)

    cutoff_summary = []
    for feature, feature_df in combined.groupby("feature"):
        pivot = feature_df.pivot_table(index="distance_km", columns="country", values="morans_i")
        mean_line = pivot.mean(axis=1, skipna=True)
        overall_cutoff = suggest_cutoff_from_series(mean_line.index.values, mean_line.values, CUTOFF_THRESHOLD)

        print(f"\n{feature}: overall (mean across countries) cutoff ≈ "
              f"{overall_cutoff:.0f} km" if pd.notna(overall_cutoff)
              else f"\n{feature}: mean never drops below threshold within {MAX_DIST_KM} km")

        for country in pivot.columns:
            country_cutoff = suggest_cutoff_from_series(pivot.index.values, pivot[country].values, CUTOFF_THRESHOLD)
            cutoff_summary.append({
                "feature": feature,
                "country": country,
                "suggested_cutoff_km": country_cutoff,
            })
        cutoff_summary.append({
            "feature": feature,
            "country": "MEAN_ACROSS_COUNTRIES",
            "suggested_cutoff_km": overall_cutoff,
        })

    pd.DataFrame(cutoff_summary).to_csv(f'{fd}/suggested_cutoffs_{version}.csv', index=False)
    print("\nSaved plots + values")

Found 34 countries in data.
  [ARG] computing correlogram (416 regions)...
  [BEL] skipped — only 2 regions with coords
  [BGR] computing correlogram (6 regions)...
  [BOL] computing correlogram (218 regions)...
  [BRA] computing correlogram (5136 regions)...
  [CAN] computing correlogram (1266 regions)...
  [CHN] computing correlogram (31 regions)...
  [DEU] computing correlogram (13 regions)...
  [ESP] computing correlogram (17 regions)...
  [FIN] skipped — only 4 regions with coords
  [FRA] computing correlogram (22 regions)...
  [GBR] computing correlogram (6 regions)...
  [GHA] computing correlogram (15 regions)...
  [GRC] skipped — only 4 regions with coords
  [HRV] skipped — only 2 regions with coords
  [HUN] skipped — only 3 regions with coords
  [IND] computing correlogram (624 regions)...
  [ITA] computing correlogram (21 regions)...
  [JPN] computing correlogram (46 regions)...
  [MEX] computing correlogram (32 regions)...
  [NGA] computing correlogram (37 regions)...
  [NZL